# Follow registers and control flow

This is Notebook 5 for the SC26 tutorial.
We will follow registers and loops through QC and QCO, compare static and executed gate counts, and explore measurement feedback.

Start with a fresh kernel and run the cells in order.
This notebook uses the same Python 3.15 environment as the rest of the tutorial.

In [ ]:
from IPython.display import Code, display
from qiskit.visualization import plot_distribution

from mqt.core.mlir import QCProgram, sample

shots = 4096
seed = 17

## Grow a Bell state into a GHZ state

For $n$ qubits, a GHZ state is $(|0\cdots0\rangle + |1\cdots1\rangle)/\sqrt{2}$.
We can build this state by applying $H$ to the first qubit, then using a loop to apply controlled-$X$ gates from it to the others.

:::{admonition} Your turn
:class: tip

For `width = 3`, predict how many controlled-$X$ gates the loop executes and which bitstrings can appear.
:::

In [ ]:
width = 3

ghz_source = f"""OPENQASM 3.0;
include "stdgates.inc";
qubit[{width}] q;
bit[{width}] result;
h q[0];
for int i in [1:{width - 1}] {{
    cx q[0], q[i];
}}
result = measure q;
"""

ghz_qc = QCProgram.from_openqasm_str(ghz_source)

OpenQASM includes both endpoints of a range, so `[1:2]` visits 1 and 2.
Our loop therefore applies `width - 1` controlled-$X$ gates.

## Inspect the register in QC

Let us see how QC represents the register and its loop:

In [ ]:
display(Code(ghz_qc.ir, language="mlir"))

For `width = 3`, `memref<3x!qc.qubit>` holds three qubit references, while `memref.load` retrieves one for a gate to use.
The loop appears as `scf.for`, whose upper bound is exclusive.

For measurement results, we use **CBit**: `qc.measure` produces a bit and `cbit.store` writes it into the output register.

## Follow the register through QCO

:::{admonition} Your turn
:class: tip

Predict what must change: if each quantum value has exactly one use, how can successive loop iterations work on the same register?
:::

In [ ]:
ghz_qco = ghz_qc.to_qco(copy=True)
display(Code(ghz_qco.ir, language="mlir"))

In QCO, **QTensor** lets us carry a whole register of quantum values through the program:

1. `qtensor.alloc` creates the register.
2. `qtensor.extract` takes out a qubit so a gate can act on it.
3. `qtensor.insert` puts the updated qubit back into the register.

The loop passes the updated register from one iteration to the next through `iter_args` and `scf.yield`.
This keeps the one-use rule for quantum values intact.

## Distinguish a loop body from executed gates

When we call `num_gates()`, we count gate operations in the IR, including measurements.
A loop body is counted just once, however many times it will run:

In [ ]:
static_gates = ghz_qc.num_gates()
static_cx = ghz_qc.num_two_qubit_gates()
print(f"Static IR: {static_gates} gates, including {static_cx} controlled-X")
print(f"This program executes: {2 * width} gates, including {width - 1} controlled-X")

Both totals include the `width` measurements.
To compare this with a circuit, let us unroll the loop:

In [ ]:
unrolled = ghz_qco.copy()
unrolled.unroll_quantum_loops()
unrolled_qc = unrolled.to_qc()
print(f"Controlled-X operations after unrolling: {unrolled_qc.num_two_qubit_gates()}")
unrolled_qc.to_qiskit(copy=True).draw("mpl")

Unrolling makes each iteration explicit while leaving the program's behavior unchanged.

## Check the observable result

Sampling this program directly in QCO should give roughly equal numbers of results with all zeros and all ones:

In [ ]:
ghz_counts = sample(ghz_qco, shots=shots, seed=seed)
plot_distribution(ghz_counts, title=f"{width}-qubit GHZ outputs")

:::{admonition} Your turn
:class: tip

Predict how the loop count and possible outcomes change for `width = 2` and `width = 4`.
Try both values in the parameter cell and rerun the GHZ section.
:::

:::{dropdown} Explain the result

The loop still contains one controlled-$X$, but it executes once for two qubits and three times for four qubits.
In either case, the outcomes are still all zeros or all ones, with equal probability.
:::

## Use a measurement to choose the next gate

:::{admonition} Your turn
:class: tip

Start with $H$, measure, and apply $X$ only if the outcome is 1.
What should a second measurement return?
:::

In [ ]:
feedback_source = """OPENQASM 3.0;
include "stdgates.inc";
qubit q;
bit outcome;
h q;
outcome = measure q;
if (outcome) {
    x q;
}
outcome = measure q;
"""

feedback_qc = QCProgram.from_openqasm_str(feedback_source)
feedback_qc.to_qiskit(copy=True).draw("mpl")

The second measurement overwrites the output bit.
Let us see how QCO represents the conditional correction:

In [ ]:
feedback_qco = feedback_qc.to_qco(copy=True)
display(Code(feedback_qco.ir, language="mlir"))

`qco.if` passes the qubit into the selected branch, which either applies $X$ or leaves it unchanged.
The branch's result then feeds the next measurement.

Unlike `qco.ctrl`, which uses a quantum control qubit, `qco.if` branches on a classical value.

In [ ]:
corrected_counts = sample(feedback_qco, shots=shots, seed=seed)
plot_distribution(corrected_counts, title="Final measurement after correction")

With the correction in place, every shot ends in zero.

:::{admonition} Your turn
:class: tip

Predict the final outcomes if we remove the conditional correction, then run the variant below.
:::

Once we remove the correction, the second measurement repeats the first result, so the outcome is again random across shots.

In [ ]:
uncorrected_source = feedback_source.replace("if (outcome) {\n    x q;\n}\n", "")
uncorrected_counts = sample(uncorrected_source, shots=shots, seed=seed)
plot_distribution([corrected_counts, uncorrected_counts], legend=["Correction", "No correction"])

:::{admonition} Read further
:class: seealso

See the [control-flow tutorial](https://mqt.readthedocs.io/projects/core/en/stable/tutorials/control_flow.html) for more detail on registers and branches.
:::